In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import pyarrow

In [19]:
print("hello")

hello


In [20]:
from pathlib import Path

folder = Path(".")
dataframes = {}

for parquet_file in sorted(folder.glob("*.parquet")):
    name = parquet_file.stem
    dataframes[name] = pd.read_parquet(parquet_file)

print(f"Loaded {len(dataframes)} parquet files")

for name, dataframe in dataframes.items():
    print(name, dataframe.shape)

Loaded 12 parquet files
brfss_2020_2024_pooled_eda (2176776, 166)
brfss_2020_2024_pooled_ml (2176776, 129)
brfss_2020_eda (401958, 293)
brfss_2020_ml (401958, 255)
brfss_2021_eda (438693, 317)
brfss_2021_ml (438693, 278)
brfss_2022_eda (445132, 342)
brfss_2022_ml (445132, 301)
brfss_2023_eda (433323, 364)
brfss_2023_ml (433323, 322)
brfss_2024_eda (457670, 316)
brfss_2024_ml (457670, 273)


## EDA

In [21]:
eda_df = dataframes["brfss_2020_2024_pooled_eda"]
eda_df.head()

,ACEDEPRS,ACEDIVRC,ACEDRINK,ACEDRUGS,ACEHURT1,ACEHVSEX,ACEPRISN,ACEPUNCH,ACESWEAR,ACETOUCH,...,_RFSMOK3,_SEX,_SMOKER3,_STATE,_STRWT,_STSTR,_TOTINDA,_URBSTAT,_WT2RAKE,_LLCPWT_POOLED
0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0,3.0,3.0,...,2.0,2.0,1.0,1.0,69.640207,11011.0,1.0,1.0,69.640207,56.867134
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2.0,NaN,1.0,69.640207,11011.0,1.0,1.0,69.640207,34.256666
2,2.0,1.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,1.0,...,1.0,2.0,4.0,1.0,279.748901,11012.0,1.0,1.0,279.748901,266.873773
3,2.0,2.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,1.0,...,1.0,2.0,4.0,1.0,69.640207,11011.0,2.0,1.0,69.640207,259.497324
4,2.0,2.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,1.0,...,1.0,2.0,4.0,1.0,69.640207,11011.0,1.0,1.0,69.640207,90.963025


In [24]:
# 1. Define the core columns we want to inspect
# (Note: column names might vary slightly depending on how it was prepped, 
# but these are standard BRFSS names)
target_cols = [
    'INCOME3',    # Household income category
    'EDUCA',      # Education level
    'GENHLTH',    # General health status (Excellent to Poor)
    'MEDCOST',    # Could not see doctor due to cost (Yes/No)
    '_LLCPWT'     # The crucial survey weight column
]

# Check which of these columns actually exist in your dataframe
existing_cols = [col for col in target_cols if col in df.columns]
print(f"Found columns: {existing_cols}")

# 2. Check the percentage of missing values for these specific columns
missing_stats = df[existing_cols].isnull().mean() * 100
print("\nPercentage of missing data per column:")
print(missing_stats)

Found columns: ['EDUCA', 'GENHLTH', '_LLCPWT']

Percentage of missing data per column:
EDUCA      0.001746
GENHLTH    0.270676
_LLCPWT    0.000000
dtype: float64


In [25]:
# Search for columns related to income or year
matches = [col for col in df.columns if 'INC' in col.upper() or 'YEAR' in col.upper()]
print(matches[:20]) # Print the first 20 matches

['IYEAR', 'SURVEY_YEAR']


In [26]:
# Check which years are present in the dataset
print("Years available:", df['IYEAR'].unique() if 'IYEAR' in df.columns else df['SURVEY_YEAR'].unique())

# Find any column related to income
income_cols = [col for col in df.columns if 'INC' in col.upper()]
print("Potential income columns:", income_cols)

Years available: <ArrowStringArray>
['2020', '2021', '2022', '2023', '2024', '2025']
Length: 6, dtype: str
Potential income columns: []


In [27]:
# Print the first 50 column names to see the naming style
print(df.columns[:50].tolist())

['ACEDEPRS', 'ACEDIVRC', 'ACEDRINK', 'ACEDRUGS', 'ACEHURT1', 'ACEHVSEX', 'ACEPRISN', 'ACEPUNCH', 'ACESWEAR', 'ACETOUCH', 'ACETTHEM', 'ADDEPEV3', 'ASTHMA3', 'ASTHNOW', 'BLIND', 'CADULT1', 'CAREGIV1', 'CASTHDX2', 'CASTHNO2', 'CCLGHOUS', 'CELLFON5', 'CHCKDNY2', 'CHECKUP1', 'CHILDREN', 'CHKHEMO3', 'CNCRAGE', 'CNCRDIFF', 'COLGHOUS', 'CRGVALZD', 'CSRVCLIN', 'CSRVCTL2', 'CSRVDEIN', 'CSRVDOC1', 'CSRVINSR', 'CSRVINST', 'CSRVPAIN', 'CSRVRTRN', 'CSRVSUM', 'CSRVTRT3', 'CSTATE1', 'CTELENM1', 'CTELNUM1', 'CVDCRHD4', 'CVDINFR4', 'CVDSTRK3', 'DEAF', 'DECIDE', 'DIABETE4', 'DIFFALON', 'DIFFDRES']


In [28]:
# 1. Search specifically for columns starting with 'INC' or containing 'INCOME'
income_cols = [col for col in df.columns if 'INC' in col.upper()]
print("Income-related columns found:", income_cols)

# 2. Check the value counts for Depression (ADDEPEV3) to see how responses are coded (e.g., 1=Yes, 2=No)
print("\nDepression (ADDEPEV3) value counts:")
print(df['ADDEPEV3'].value_counts(dropna=False))

Income-related columns found: []

Depression (ADDEPEV3) value counts:
ADDEPEV3
2.0    1727191
1.0     436902
NaN      12683
Name: count, dtype: int64


In [29]:
# 1. Broader search for income, earnings, or financial variables
financial_cols = [col for col in df.columns if any(term in col.upper() for term in ['INC', 'MONEY', 'EARN', 'FINAN'])]
print("Financial/Income related columns found:", financial_cols[:10])

# 2. Calculate the percentage of depression (ADDEPEV3 == 1) by Year
# (Assuming 'IYEAR' is your year column)
year_col = 'IYEAR' if 'IYEAR' in df.columns else 'SURVEY_YEAR'

# Clean data: filter out NaNs and map 1=Yes, 2=No
df_clean = df.dropna(subset=[year_col, 'ADDEPEV3']).copy()
depression_trend = df_clean.groupby(year_col)['ADDEPEV3'].apply(lambda x: (x == 1.0).mean() * 100)

print("\nDepression Rate (%) by Year:")
print(depression_trend)

Financial/Income related columns found: []

Depression Rate (%) by Year:
IYEAR
2020    18.954833
2021    19.539067
2022    20.664658
2023    20.519019
2024    21.037416
2025    20.781491
Name: ADDEPEV3, dtype: float64


In [30]:
eda_df.columns

Index(['ACEDEPRS', 'ACEDIVRC', 'ACEDRINK', 'ACEDRUGS', 'ACEHURT1', 'ACEHVSEX',
       'ACEPRISN', 'ACEPUNCH', 'ACESWEAR', 'ACETOUCH',
       ...
       '_RFSMOK3', '_SEX', '_SMOKER3', '_STATE', '_STRWT', '_STSTR',
       '_TOTINDA', '_URBSTAT', '_WT2RAKE', '_LLCPWT_POOLED'],
      dtype='str', length=166)

### Checking Missing Values

In [ ]:
missing = (
    eda_df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")

)

missing["missing_percent"] = (
    missing["missing_count"] / len(eda_df) * 100

)

missing.head(20)

,missing_count,missing_percent
COLGHOUS,2176678,99.995498
CSRVCTL2,2171727,99.768051
CCLGHOUS,2170440,99.708927
CSRVINST,2152183,98.870210
CASTHNO2,2151297,98.829507
CSRVCLIN,2143790,98.484640
CSRVDEIN,2143764,98.483445
CSRVINSR,2143740,98.482343
CSRVRTRN,2143682,98.479678
CSRVSUM,2143638,98.477657


### Checking Duplicate Rows and Examining Duplicate Rows

In [ ]:
eda_df.duplicated().sum()

np.int64(0)

In [ ]:
for column in eda_df.select_dtypes(include="object").columns:
    print(f"\n{column}")
    print(eda_df[column].value_counts(dropna=False).head(10))


IDATE
IDATE
12202022    2169
01032023    2151
01042023    2100
12282022    2077
12212022    2070
06042024    2047
03172020    2032
06252024    2020
03122020    2018
06242024    2010
Name: count, dtype: int64

IDAY
IDAY
09    77815
08    77173
16    77100
15    76756
13    76606
11    76385
10    76290
14    76211
06    76051
07    76041
Name: count, dtype: int64

IMONTH
IMONTH
03    203736
12    193332
08    190937
11    187387
10    184086
06    183899
07    181117
05    179617
04    175851
02    173904
Name: count, dtype: int64

IYEAR
IYEAR
2024    463428
2022    442981
2023    433671
2021    427317
2020    389826
2025     19553
Name: count, dtype: int64

SEQNO
SEQNO
2022000001    54
2022000002    54
2022000003    54
2022000004    54
2022000005    54
2022000006    54
2022000007    54
2022000008    54
2022000009    54
2022000010    54
Name: count, dtype: int64


/var/folders/1y/_y3kcj8n1z7c3fn135fdlnmw0000gn/T/ipykernel_76960/3347053875.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in eda_df.select_dtypes(include="object").columns:
